# Nieuw-West on the map

The 1 m results for all of Nieuw-West on the reference hot day (1 July 2015) in an interactive leafmap view: afternoon felt temperature (PET), shade at 15:00, neighbourhoods, the pedestrian main routes and the proposed trees. Run `make_map.py`, `district_stats.py`, `check_district.py` and `plan_trees.py` first (see the README).

Open with `uv run jupyter lab notebooks/nieuw_west_leafmap.ipynb`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import leafmap
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN = ROOT / "data" / "processed" / "districts" / "nieuw-west_2015-07-01"
CITY = ROOT / "data" / "raw" / "city"
TREES = ROOT / "data" / "processed" / "trees"


def rd(path):
    """Dutch files often hold RD coordinates without saying so."""
    g = gpd.read_file(path)
    return g.set_crs(28992, allow_override=True) if g.total_bounds[0] > 1000 else g.to_crs(28992)

In [ ]:
stats = pd.read_csv(RUN / "buurt_stats.csv")
buurten = rd(CITY / "buurten_nieuw-west.geojson").merge(stats, on="code", suffixes=("", "_s"))
routes = rd(CITY / "ams_plushoofdnetten.geojson")
routes = routes[routes["VOET"].isin(["PLUS", "HOOFD"])].clip(buurten.union_all())
trees = pd.concat([rd(p).assign(plan=p.parent.name) for p in sorted(TREES.glob("*/trees.geojson"))])

In [ ]:
m = leafmap.Map(center=[52.36, 4.81], zoom=14, draw_control=False)
m.add_basemap("nlmaps.grijs")
m.add_raster(str(RUN / "pet_afternoon_cog.tif"), colormap="magma", vmin=25, vmax=45,
             layer_name="Felt temperature (PET), 12:00 to 18:00")
m.add_raster(str(RUN / "shade_15h_cog.tif"), colormap="Greys", vmin=0, vmax=1.3, layer_name="Shade at 15:00",
             visible=False)
m.add_gdf(buurten[["naam", "pet_mean", "walk_shade_15h", "homes_near_park", "geometry"]].to_crs(4326),
          layer_name="Neighbourhoods", style={"color": "#1d2321", "weight": 1, "fillOpacity": 0})
m.add_gdf(routes[["STT_NAAM", "VOET", "geometry"]].to_crs(4326), layer_name="Pedestrian main routes",
          style={"color": "#0d6b62", "weight": 2.5})
m.add_gdf(trees[["plan", "rank", "geometry"]].to_crs(4326), layer_name="Proposed trees")
m.add_colorbar(colors=["#000004", "#51127c", "#b73779", "#fc8961", "#fcfdbf"], vmin=25, vmax=45,
               caption="PET (C)")
m

In [ ]:
cols = ["naam", "residents", "aged_65_plus", "pet_mean", "extreme_share", "walk_shade_15h", "homes_near_park"]
stats[stats["residents"] >= 1000][cols].sort_values("walk_shade_15h").round(2).head(15)

Credits: AHN4, BAG and BGT (PDOK); Gemeente Amsterdam (infrarood2023, bomen, plus- en hoofdnetten, gebieden), CC BY 4.0; CBS; KNMI. Model: SOLWEIG-GPU.